In [4]:
from pathlib import Path
import re
import pandas as pd

In [18]:
# 경로 설정
base_dir = Path(
    r"C:\Users\user\Desktop\KDT_2026\프로젝트\02_데이터_프로젝트"
    r"\02_csv\02_instagram"
)

raw_dir = base_dir / "raw"
output_path = base_dir / "instagram3.csv"

csv_files = sorted(raw_dir.glob("*.csv"))

print(f"원본 CSV 개수: {len(csv_files)}")

원본 CSV 개수: 13


In [19]:
# CSV 취합
dataframes = []
source_summary = []

for file_path in csv_files:
    # 예: instagram_갑옷씨_final.csv → 갑옷씨
    match = re.match(r"^instagram_([^_]+)", file_path.stem)

    if match is None:
        raise ValueError(f"파일명에서 keyword를 추출할 수 없습니다: {file_path.name}")

    keyword = match.group(1)

    df = pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        dtype={
            "id": "str",
            "like_count": "Int64",
            "comments_count": "Int64",
        }
    )

    df["keyword"] = keyword
    dataframes.append(df)

    source_summary.append(
        {
            "파일명": file_path.name,
            "keyword": keyword,
            "행 수": len(df),
        }
    )

instagram_df = pd.concat(
    dataframes,
    ignore_index=True,
    sort=False,
)

# count 컬럼을 정수형으로 변환
count_columns = ["like_count", "comments_count"]

for column in count_columns:
    instagram_df[column] = pd.to_numeric(
        instagram_df[column],
        errors="coerce",
    ).astype("Int64")

# timestamp 컬럼을 datetime 형식으로 변환
instagram_df["timestamp"] = (
    pd.to_datetime(
        instagram_df["timestamp"],
        format="%Y-%m-%dT%H:%M:%S%z",
        errors="coerce",
        utc=True,
    )
    .dt.tz_convert("Asia/Seoul")
)

print(instagram_df.dtypes)

print(f"통합 행 수: {len(instagram_df):,}")
print(f"컬럼: {instagram_df.columns.tolist()}")

id                                       str
caption                                  str
media_type                               str
permalink                                str
timestamp         datetime64[us, Asia/Seoul]
like_count                             Int64
comments_count                         Int64
keyword                                  str
dtype: object
통합 행 수: 20,335
컬럼: ['id', 'caption', 'media_type', 'permalink', 'timestamp', 'like_count', 'comments_count', 'keyword']


In [20]:
instagram_df.head()

,id,caption,media_type,permalink,timestamp,like_count,comments_count,keyword
0,17898709716673599,#먼작귀 #치이카와 #코스프레 #갑옷씨 #포셰트갑옷씨 #포쉐트갑옷씨 #수원 #수원메...,CAROUSEL_ALBUM,https://www.instagram.com/p/De9AmC6mlSY/,2026-10-26 18:55:23+09:00,15,1,갑옷씨
1,17878685700651547,고블린과 갑옷씨💟\r\n치킨든 고블린과 쫌 더커진 고블린까지〰️\r\n갑옷아저씨도 ...,CAROUSEL_ALBUM,https://www.instagram.com/p/De_o-1lEwRB/,2026-10-27 19:26:47+09:00,4,2,갑옷씨
2,18103745312140350,🌱뭔가 작고 귀엽다\r\n\r\n지치고 팍팍한 일상 속에서 요즘 에디터가 가장 위로...,CAROUSEL_ALBUM,https://www.instagram.com/p/DazEHTlAWLR/,2026-07-15 12:08:10+09:00,18083,54,갑옷씨
3,18290126140292291,맥도날드 레스토랑(?)에 온 치이와 포쉐트갑옷씨\r\nちいかわレストランじゃなくて、マ...,IMAGE,https://www.instagram.com/p/De87JZHKcxr/,2026-10-26 18:08:48+09:00,27,6,갑옷씨
4,17924891337200386,먼작귀 토벌 전🍀\r\n\r\n홍대 미니소를 갔는데🐢\r\n\r\n역시나 한개도 못...,CAROUSEL_ALBUM,https://www.instagram.com/p/DeV7BvbE1PG/,2026-10-11 14:36:21+09:00,<NA>,0,갑옷씨


In [21]:
# 결과 확인
summary_df = pd.DataFrame(source_summary)
display(summary_df)

display(instagram_df.head())
display(instagram_df.tail())

instagram_df["keyword"].value_counts()

,파일명,keyword,행 수
0,instagram_갑옷씨_final.csv,갑옷씨,58
1,instagram_데카츠요_final.csv,데카츠요,37
2,instagram_랏코_final.csv,랏코,1031
3,instagram_먼작귀_261027_2587.csv,먼작귀,2587
4,instagram_모몽가_final.csv,모몽가,2926
5,instagram_시사_final.csv,시사,1038
6,instagram_아노코_final.csv,아노코,208
7,instagram_우사기_final.csv,우사기,2917
8,instagram_치이카와_final.csv,치이카와,2928
9,instagram_카니_final.csv,카니,918


,id,caption,media_type,permalink,timestamp,like_count,comments_count,keyword
0,17898709716673599,#먼작귀 #치이카와 #코스프레 #갑옷씨 #포셰트갑옷씨 #포쉐트갑옷씨 #수원 #수원메...,CAROUSEL_ALBUM,https://www.instagram.com/p/De9AmC6mlSY/,2026-10-26 18:55:23+09:00,15,1,갑옷씨
1,17878685700651547,고블린과 갑옷씨💟\r\n치킨든 고블린과 쫌 더커진 고블린까지〰️\r\n갑옷아저씨도 ...,CAROUSEL_ALBUM,https://www.instagram.com/p/De_o-1lEwRB/,2026-10-27 19:26:47+09:00,4,2,갑옷씨
2,18103745312140350,🌱뭔가 작고 귀엽다\r\n\r\n지치고 팍팍한 일상 속에서 요즘 에디터가 가장 위로...,CAROUSEL_ALBUM,https://www.instagram.com/p/DazEHTlAWLR/,2026-07-15 12:08:10+09:00,18083,54,갑옷씨
3,18290126140292291,맥도날드 레스토랑(?)에 온 치이와 포쉐트갑옷씨\r\nちいかわレストランじゃなくて、マ...,IMAGE,https://www.instagram.com/p/De87JZHKcxr/,2026-10-26 18:08:48+09:00,27,6,갑옷씨
4,17924891337200386,먼작귀 토벌 전🍀\r\n\r\n홍대 미니소를 갔는데🐢\r\n\r\n역시나 한개도 못...,CAROUSEL_ALBUM,https://www.instagram.com/p/DeV7BvbE1PG/,2026-10-11 14:36:21+09:00,<NA>,0,갑옷씨


,id,caption,media_type,permalink,timestamp,like_count,comments_count,keyword
20330,18123750298307874,⭐️ 소문내기 이벤트 안내 ⭐️\r\n\r\n(슬쩍- 담당자 사심채운)\r\n할미슈...,CAROUSEL_ALBUM,https://www.instagram.com/p/CwsNk5cLqBe/,2023-09-02 22:17:36+09:00,145,22,하치와레
20331,18173523367274769,".\r\n에어팟, 차키, 카드, 머리끈등\r\n자그마한 걸 넣기 좋은\r\n💗치이카...",CAROUSEL_ALBUM,https://www.instagram.com/p/CqkYMHjLnxY/,2023-04-03 18:08:03+09:00,132,3,하치와레
20332,17962570739304285,新しい友達歓迎式,CAROUSEL_ALBUM,https://www.instagram.com/p/Cl6aSTqP2nM/,2022-12-09 00:52:42+09:00,146,2,하치와레
20333,18029555146666346,😈\r\n\r\n#가챠 #가챠샵 #피규어 #키덜트 #토이 #gacha #figure...,IMAGE,https://www.instagram.com/p/CzyV3yUL2_d/,2023-11-18 20:59:43+09:00,148,20,하치와레
20334,17944972280444947,헤드드레스 만들어줬ㄸㅏ🧵🪡,CAROUSEL_ALBUM,https://www.instagram.com/p/CmYMVynrYbn/,2022-12-20 14:28:03+09:00,271,4,하치와레


keyword
하치와레      2932
치이카와      2928
모몽가       2926
우사기       2917
쿠리만쥬      2719
먼작귀       2587
시사        1038
랏코        1031
카니         918
아노코        208
갑옷씨         58
데카츠요        37
파자마파티즈      36
Name: count, dtype: int64

In [22]:
# instagram.csv 저장
# utf-8-sig는 Excel에서 한글이 깨지는 것을 방지합니다.
instagram_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"저장 완료: {output_path}")

저장 완료: C:\Users\user\Desktop\KDT_2026\프로젝트\02_데이터_프로젝트\02_csv\02_instagram\instagram3.csv


In [23]:
check_df = pd.read_csv(
    output_path,
    encoding="utf-8-sig"
)

assert len(check_df) == len(instagram_df)
assert check_df.columns.tolist() == instagram_df.columns.tolist()
assert check_df["keyword"].value_counts().to_dict() == (
    instagram_df["keyword"].value_counts().to_dict()
)

print(f"검증 완료: {len(check_df):,}행, {len(check_df.columns)}개 컬럼")

검증 완료: 20,335행, 8개 컬럼


In [24]:
instagram_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20335 entries, 0 to 20334
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype                     
---  ------          --------------  -----                     
 0   id              20335 non-null  str                       
 1   caption         20154 non-null  str                       
 2   media_type      20335 non-null  str                       
 3   permalink       20335 non-null  str                       
 4   timestamp       20335 non-null  datetime64[us, Asia/Seoul]
 5   like_count      17475 non-null  Int64                     
 6   comments_count  20335 non-null  Int64                     
 7   keyword         20335 non-null  str                       
dtypes: Int64(2), datetime64[us, Asia/Seoul](1), str(5)
memory usage: 12.5 MB
